##  Installation

In [ ]:
!pip install -q pyannote.audio
!pip install -q torch torchaudio
!pip install -q pandas tqdm

##  Imports

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*torchcodec is not installed correctly.*")

In [2]:
import os
import json
import warnings
from pathlib import Path
from typing import List, Dict, Optional

import torch
import torchaudio
import pandas as pd
from tqdm.auto import tqdm

from pyannote.audio import Pipeline
from pyannote.core import Annotation, Segment

warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.9.0+cu126
CUDA available: True
CUDA device: Tesla P100-PCIE-16GB


In [ ]:
HF_TOKEN = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"


USE_GPU = True 
DEVICE = torch.device("cuda" if USE_GPU and torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


MIN_SPEAKERS = None 
MAX_SPEAKERS = None

OUTPUT_DIR = "./output"
OUTPUT_FILENAME = "submission.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)

Using device: cuda


##  Load Audio Files

In [4]:
DATA_DIR = "/kaggle/input/competitions/dl-sprint-4-0-bengali-speaker-diarization-challenge/diarization/diarization/test/audio" 


AUDIO_EXTENSIONS = ('.wav', '.mp3', '.flac', '.m4a', '.ogg', '.opus')

audio_paths = sorted([
    str(p) for p in Path(DATA_DIR).rglob('*') 
    if p.suffix.lower() in AUDIO_EXTENSIONS
])

filenames = [Path(p).stem for p in audio_paths]

print(f"\n Found {len(audio_paths)} audio files")
print(f"Directory: {DATA_DIR}")

if len(audio_paths) > 0:
    print(f"\n Sample files:")
    for i, path in enumerate(audio_paths[:5]):
        print(f"   {i+1}. {Path(path).name}")
    if len(audio_paths) > 5:
        print(f"   ... and {len(audio_paths) - 5} more")
else:
    print("\n WARNING: No audio files found!")
    print("   Please check your DATA_DIR path.")


 Found 14 audio files
Directory: /kaggle/input/competitions/dl-sprint-4-0-bengali-speaker-diarization-challenge/diarization/diarization/test/audio

 Sample files:
   1. test_010.wav
   2. test_012.wav
   3. test_016.wav
   4. test_018.wav
   5. test_019.wav
   ... and 9 more


##  Load Pyannote Pipeline

In [5]:
def load_pyannote_pipeline(hf_token: str, use_gpu: bool = True) -> Optional[Pipeline]:
    
    try:
        print("\n Downloading models (first time only)")
        pipeline = Pipeline.from_pretrained(
            "pyannote/speaker-diarization-3.1",
            token=hf_token
        )
        
        if use_gpu and torch.cuda.is_available():
            print("\n Moving pipeline to GPU")
            pipeline.to(torch.device("cuda"))
            print("Pipeline running on GPU")
        else:
            print("\n Pipeline running on CPU")
        
        print("\nPyannote pipeline loaded successfully!")
        return pipeline
        
    except Exception as e:
        print(f"\n Error loading pipeline: {e}\n")
        return None


pipeline = load_pyannote_pipeline(HF_TOKEN, use_gpu=USE_GPU)

config.yaml:   0%|          | 0.00/469 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

plda/xvec_transform.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

plda/plda.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/26.6M [00:00<?, ?B/s]


 Moving pipeline to GPU
Pipeline running on GPU

Pyannote pipeline loaded successfully!


##  Diarization Function

In [6]:
def diarize_audio(audio_path, pipeline, min_speakers=None, max_speakers=None):

    waveform, sample_rate = torchaudio.load(audio_path)

    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(sample_rate, 16000)
        waveform = resampler(waveform)
        sample_rate = 16000

    output = pipeline(
        {"waveform": waveform, "sample_rate": sample_rate},
        min_speakers=min_speakers,
        max_speakers=max_speakers,
        segmentation_threshold=0.5,  
        clustering_threshold=0.7    
    )

    diarization = output.speaker_diarization

    segments = []
    for segment, _, speaker in diarization.itertracks(yield_label=True):
        segments.append({
            "start_time": float(segment.start),
            "end_time": float(segment.end),
            "speaker_id": speaker
        })

    return segments


## 

## Overlap Handling

In [7]:
def resolve_overlaps_advanced(segments, min_duration=0.3):
    if not segments:
        return []
    
    segments = sorted(segments, key=lambda x: x["start_time"])
    
    resolved = []
    
    for seg in segments:
        current_seg = {
            "start_time": float(seg["start_time"]),
            "end_time": float(seg["end_time"]),
            "speaker_id": seg["speaker_id"]
        }
        
        if not resolved:
            if current_seg["end_time"] - current_seg["start_time"] >= min_duration:
                resolved.append(current_seg)
            continue
        
        prev = resolved[-1]
        
        if current_seg["start_time"] >= prev["end_time"]:
            if current_seg["end_time"] - current_seg["start_time"] >= min_duration:
                resolved.append(current_seg)
        else:
            
            if current_seg["end_time"] <= prev["end_time"]:
                continue
            
            current_seg["start_time"] = prev["end_time"]
            
            duration = current_seg["end_time"] - current_seg["start_time"]
            if duration >= min_duration:
                resolved.append(current_seg)
    
    merged = merge_same_speaker_segments(resolved, max_gap=0.3)
    
    return merged


def merge_same_speaker_segments(segments, max_gap=0.3):
    if not segments:
        return []
    
    merged = []
    
    for seg in segments:
        if not merged:
            merged.append(seg.copy())
            continue
        
        prev = merged[-1]
        
        if (prev["speaker_id"] == seg["speaker_id"] and 
            seg["start_time"] - prev["end_time"] <= max_gap):
            prev["end_time"] = seg["end_time"]
        else:
            merged.append(seg.copy())
    
    return merged

##  Process All Files

In [8]:
if pipeline is None:
    print("\n Cannot process files - pipeline not loaded.")
    print("   Please fix the authentication issues above.\n")
elif len(audio_paths) == 0:
    print("\n Cannot process files - no audio files found.")
    print("   Please check your DATA_DIR path.\n")
else:
    print("\n" + "="*70)
    print(f"Processing {len(audio_paths)} audio files")
    print("="*70 + "\n")
    
    results = []
    
    for filename, audio_path in tqdm(
        list(zip(filenames, audio_paths)),
        desc="  Diarizing",
        unit="file"
    ):
        try:
            segments = diarize_audio(
                audio_path,
                pipeline,
                min_speakers=MIN_SPEAKERS,
                max_speakers=MAX_SPEAKERS
            )

            segments = resolve_overlaps_advanced(segments, min_duration=0.3)
            results.append({
                'filename': filename,
                'diarization': json.dumps(segments, ensure_ascii=False)
            })
            
        except Exception as e:
            print(f"\n Error processing {filename}: {e}")
            results.append({
                'filename': filename,
                'diarization': json.dumps([], ensure_ascii=False)
            })
    
    print(f"\n Processing complete: {len(results)} files processed")


Processing 14 audio files



  Diarizing:   0%|          | 0/14 [00:00<?, ?file/s]


 Processing complete: 14 files processed


##  Save Results

In [9]:
if 'results' in locals() and len(results) > 0:
    df = pd.DataFrame(results)
    
    output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)
    df.to_csv(output_path, index=False)
    
    print("\n" + "="*70)
    print("Results Saved")
    print("="*70)
    print(f"\n Output file: {output_path}")
    print(f" Total files: {len(df)}")
    
    print("\n Sample results:\n")
    display(df.head(10))
    
    print("\n" + "="*70)
    print("Statistics")
    print("="*70 + "\n")
    
    total_segments = 0
    speaker_counts = []
    successful_files = 0
    
    for diar_json in df['diarization']:
        segments = json.loads(diar_json)
        if segments:
            successful_files += 1
            total_segments += len(segments)
            speakers = set(seg['speaker_id'] for seg in segments)
            speaker_counts.append(len(speakers))
    
    if speaker_counts:
        import numpy as np
        print(f" Successfully processed: {successful_files}/{len(df)} files")
        print(f"\n Segments:")
        print(f"   Total: {total_segments}")
        print(f"   Average per file: {total_segments / len(df):.1f}")
        print(f"\n Speakers:")
        print(f"   Average per file: {np.mean(speaker_counts):.1f}")
        print(f"   Range: {min(speaker_counts)} - {max(speaker_counts)}")
    else:
        print("  No segments detected in any file")
    
    print("\n" + "="*70 + "\n")
else:
    print("\n  No results to save")


Results Saved

 Output file: ./output/submission.csv
 Total files: 14

 Sample results:



,filename,diarization
0,test_010,"[{""start_time"": 13.345343750000001, ""end_time""..."
1,test_012,"[{""start_time"": 0.9422187500000001, ""end_time""..."
2,test_016,"[{""start_time"": 35.73846875, ""end_time"": 36.68..."
3,test_018,"[{""start_time"": 0.87471875, ""end_time"": 4.8234..."
4,test_019,"[{""start_time"": 2.32596875, ""end_time"": 4.0809..."
5,test_020,"[{""start_time"": 4.45221875, ""end_time"": 11.354..."
6,test_021,"[{""start_time"": 2.29221875, ""end_time"": 4.2665..."
7,test_022,"[{""start_time"": 3.8109687500000002, ""end_time""..."
8,test_023,"[{""start_time"": 0.03096875, ""end_time"": 0.3515..."
9,test_024,"[{""start_time"": 0.03096875, ""end_time"": 1.8872..."



Statistics

 Successfully processed: 14/14 files

 Segments:
   Total: 12043
   Average per file: 860.2

 Speakers:
   Average per file: 14.4
   Range: 1 - 30




## Evaluation

In [10]:
# import pandas as pd
# import json
# import os
# from pyannote.core import Annotation, Segment
# from pyannote.metrics.diarization import DiarizationErrorRate

# GT_DIR = "/kaggle/input/datasets/mirazul687/annotation-diarization/annotation"

# metric = DiarizationErrorRate(collar=0.25, skip_overlap=True)

# def time_to_seconds(t):
#     h, m, s = map(int, t.split(":"))
#     return h * 3600 + m * 60 + s


# def csv_to_annotation(csv_path):
#     df = pd.read_csv(csv_path)
#     annotation = Annotation()

#     for _, row in df.iterrows():
#         start = time_to_seconds(row["start_time"])
#         end = time_to_seconds(row["end_time"])
#         speaker = str(row["speaker_id"])

#         segment = Segment(float(start), float(end))
#         annotation[segment] = speaker

#     return annotation



# def json_to_annotation(json_string):
#     segments = json.loads(json_string)
#     annotation = Annotation()

#     for seg in segments:
#         segment = Segment(seg["start_time"], seg["end_time"])
#         annotation[segment] = str(seg["speaker_id"])
#     return annotation


# pred_df = pd.read_csv("./output/submission.csv")

# total_der = 0
# count = 0

# for _, row in pred_df.iterrows():
#     filename = row["filename"]

#     gt_path = os.path.join(GT_DIR, f"{filename}.csv")

#     if not os.path.exists(gt_path):
#         print("Missing GT:", gt_path)
#         continue

#     reference = csv_to_annotation(gt_path)
#     hypothesis = json_to_annotation(row["diarization"])

#     der = metric(reference, hypothesis)

#     print(f"{filename} DER: {der:.4f}")

#     total_der += der
#     count += 1


# if count > 0:
#     print(f"Average DER: {total_der / count:.4f}")
# else:
#     print("No files evaluated.")
